### Base model

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('seattle-weather.csv')

In [3]:
df = df.drop(['date','weather'],axis=1)

In [4]:
df.head()

,precipitation,temp_max,temp_min,wind
0,0.0,12.8,5.0,4.7
1,10.9,10.6,2.8,4.5
2,0.8,11.7,7.2,2.3
3,20.3,12.2,5.6,4.7
4,1.3,8.9,2.8,6.1


In [5]:
X = df.drop('wind',axis=1)
Y = df['wind']

In [ ]:
X_1 = np.array(df['temp_max'])
Y_1 = df['wind']

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

In [7]:
X_train,X_test,Y_train,Y_test = train_test_split(X,Y, test_size=.15)
model = LinearRegression()
model.fit(X_train,Y_train)
mean_absolute_error(Y_test, model.predict(X_test))

1.166972066966548

In [ ]:
mean_absolute_error(Y_test, model.predict(X_test)) / df['wind'].mean()

0.3447440577663526

In [ ]:
model.coef_

array([ 0.06396953, -0.03641586,  0.03100551])

### Polynomial Features

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

In [ ]:
poly = PolynomialFeatures(20)
poly_X = poly.fit_transform(X)

In [ ]:
poly_X.shape

(1461, 1771)

In [ ]:
X_train,X_test,Y_train,Y_test = train_test_split(poly_X,Y, test_size=.15)
model_p = LinearRegression()
model_p.fit(X_train,Y_train)
mean_absolute_error(Y_test, model_p.predict(X_test))

1869424.641348215

In [ ]:
model_p.coef_

array([ 5.39884003e-31,  2.81395470e-33,  8.17696537e-34, ...,
       -2.63007123e-29,  1.66523409e-29,  7.59508588e-29])

### L1 Regularization: Lasso

In [ ]:
from sklearn.linear_model import Lasso, Ridge

In [ ]:
X_train,X_test,Y_train,Y_test = train_test_split(poly_X,Y, test_size=.15)
model_l = Lasso(alpha=5, max_iter=1000)
model_l.fit(X_train,Y_train)
mean_absolute_error(Y_test, model_l.predict(X_test))

/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.073e+03, tolerance: 2.542e-01
  model = cd_fast.enet_coordinate_descent(


1.0721182927408826

In [ ]:
model_l.coef_

array([ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
        1.68496194e-34, -6.54110569e-33, -3.09022082e-32])

### L2 Regularization: Ridge

In [ ]:
X_train,X_test,Y_train,Y_test = train_test_split(poly_X,Y, test_size=.15)
model_r = Ridge(alpha=5)
model_r.fit(X_train,Y_train)
mean_absolute_error(Y_test, model_r.predict(X_test))

/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_ridge.py:255: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(


1562.0976279729837

In [ ]:
model_r.coef_

array([ 0.00000000e+00,  1.05787459e-70, -1.79195717e-71, ...,
       -4.87363152e-41, -2.81763018e-41, -1.62315772e-41])

### Elastic NET

In [ ]:
from sklearn.linear_model import ElasticNet

In [ ]:
X_train,X_test,Y_train,Y_test = train_test_split(poly_X,Y, test_size=.15)
model_e = ElasticNet(alpha=5)
model_e.fit(X_train,Y_train)
mean_absolute_error(Y_test, model_e.predict(X_test))

/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.065e+03, tolerance: 2.554e-01
  model = cd_fast.enet_coordinate_descent(


1.1520258269059336

In [ ]:
poly_X.shape

(1461, 3276)

### Cross-validation

In [ ]:
from sklearn.linear_model import LassoCV

In [ ]:
poly_X.shape

(1461, 11)

In [ ]:
scaler = StandardScaler()
scaled_X = scaler.fit_transform(poly_X)

In [ ]:
X_train,X_test,Y_train,Y_test = train_test_split(scaled_X,Y, test_size=.15)
model_lcv = LassoCV(alphas=[1,3,5,10,15,20])
model_lcv.fit(X_train,Y_train)
mean_absolute_error(Y_test, model_lcv.predict(X_test))

1.2360208043366785

In [ ]:
model_lcv.get_params()

{'alphas': [1, 3, 5, 10, 15, 20],
 'copy_X': True,
 'cv': None,
 'eps': 0.001,
 'fit_intercept': True,
 'max_iter': 1000000,
 'n_alphas': 100,
 'n_jobs': None,
 'positive': False,
 'precompute': 'auto',
 'random_state': None,
 'selection': 'cyclic',
 'tol': 0.0001,
 'verbose': False}

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
pipe = Pipeline(
    [
        ('Scaler', StandardScaler()),
        ('LR', ElasticNet(alpha=5))
    ]
)

In [ ]:
X_train,X_test,Y_train,Y_test = train_test_split(poly_X,Y, test_size=.15)

In [ ]:
pipe.fit(X_train,Y_train)

Pipeline(steps=[('Scaler', StandardScaler()), ('LR', ElasticNet(alpha=5))])

In [ ]:
mean_absolute_error(Y_test, pipe.predict(X_test))

1.1268753204893414